# LaTeX Symbol Demo

Demonstration of the `latex_symbol` functionality on `Gate`, `FidelityIndex`, and `Path`.

In [1]:
from IPython.display import Math, display
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import Clifford, QubitSparsePauli

from qiskit_noise_learning.gate_sets import ModelGate, ModelGateSet
from qiskit_noise_learning.models import CompleteFidelityModel
from qiskit_noise_learning.sequences import FidelityIndex, Path

## Gate Setup

Create a simple 2-qubit gate set with a custom LaTeX symbol for the CZ gate.

In [2]:
gate_set = ModelGateSet(2)

ident_1q = Clifford(QuantumCircuit(1))
gate_set.add_gate(ModelGate("P", [((0,), ident_1q)], prep_idxs=[0, 1], qubit_idxs=[0, 1]))
gate_set.add_gate(ModelGate("M", [((0,), ident_1q)], meas_idxs=[0, 1], qubit_idxs=[0, 1]))

# CZ gate (identity Clifford for demo purposes) with a custom latex_symbol
ident_2q = Clifford(QuantumCircuit(2))
gate_set.add_gate(
    ModelGate("CZ", [((0, 1), ident_2q)], qubit_idxs=[0, 1], latex_symbol=r"\mathrm{CZ}")
)

print(f"Gate 'CZ' latex_symbol: {gate_set['CZ'].latex_symbol}")
print(f"Gate 'P' latex_symbol (default): {gate_set['P'].latex_symbol}")

Gate 'CZ' latex_symbol: \mathrm{CZ}
Gate 'P' latex_symbol (default): P


## Fidelity Index Labels

Create fidelity indices and display them in both formats.

In [3]:
model = CompleteFidelityModel(gate_set)

fi_x0 = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli.from_sparse_label(("X", [0]), num_qubits=2),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset(),
)

fi_z1 = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli.from_sparse_label(("Z", [1]), num_qubits=2),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset(),
)

fi_prep = FidelityIndex.from_gate(
    gate=gate_set["P"],
    pauli=QubitSparsePauli.identity(num_qubits=2),
    in_bit_indices=frozenset(),
    out_bit_indices=frozenset({0}),
)

fi_meas = FidelityIndex.from_gate(
    gate=gate_set["M"],
    pauli=QubitSparsePauli.identity(num_qubits=2),
    in_bit_indices=frozenset({0}),
    out_bit_indices=frozenset(),
)

# Transition format
display(Math(model.fidelity_index_latex_symbol(fi_x0, format="transition")))
display(Math(model.fidelity_index_latex_symbol(fi_z1, format="transition")))
display(Math(model.fidelity_index_latex_symbol(fi_prep, format="transition")))
display(Math(model.fidelity_index_latex_symbol(fi_meas, format="transition")))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [4]:
# Index format
display(Math(model.fidelity_index_latex_symbol(fi_x0, format="index")))
display(Math(model.fidelity_index_latex_symbol(fi_z1, format="index")))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Path Labels

Create paths and display their LaTeX representations.

In [5]:
# Simple path with just a repeatable fragment
path = Path(
    start_fragment=[fi_prep],
    repeatable_fragment=[fi_x0, fi_z1],
    end_fragment=[fi_meas],
    depth=5,
)
display(Math(model.path_latex_symbol(path)))

<IPython.core.display.Math object>

In [6]:
# Unbound path (depth=None shows 'd')
path_unbound = Path(
    start_fragment=[fi_prep],
    repeatable_fragment=[fi_x0],
    end_fragment=[fi_meas],
    depth=None,
)
display(Math(model.path_latex_symbol(path_unbound)))

<IPython.core.display.Math object>

In [7]:
# Path with start and end fragments
path_full = Path(
    start_fragment=[fi_prep],
    repeatable_fragment=[fi_x0, fi_z1],
    end_fragment=[fi_meas],
    depth=3,
)
display(Math(model.path_latex_symbol(path_full)))

<IPython.core.display.Math object>

In [8]:
# Path with index format
display(Math(model.path_latex_symbol(path_full, format="index")))

<IPython.core.display.Math object>

# To do:

- Make custom `format="index"` behaviour for `PauliFidelityModel` that simplifies to the standard that people write.
- There should be an option to drop the start/end fragment from unbound paths (should this be default?)